# scGPT Fine-tuning for MAPT Tangle Stage Prediction

**Project:** Fine-tuning scGPT, a transformer-based single-cell foundation model, to predict neurofibrillary tangle stage (Braak staging) from single-nucleus RNA-seq data of Alzheimer's disease neurons.

**Input:** `neurons_hvg_preprocessed.h5ad` — output of `scgpt_preprocessing.ipynb`  
**Model:** scGPT whole-human (pretrained on 33 million normal human cells)  
**Task:** 4-class classification — Tangle Stage 1, 2, 5, 6

---
**Pipeline overview:**
1. Setup and environment configuration
2. Load preprocessed data and model configuration
3. Tokenize gene expression for scGPT input format
4. Build dataloaders with class imbalance handling
5. Load pretrained scGPT and configure fine-tuning strategy
6. Train with weighted loss and early stopping
7. Evaluate on held-out test set
8. Visualize scGPT embeddings and save results

---
**Environment:** Run using the `scgpt_mapt` conda environment on NEU HPC
```bash
unset PYTHONPATH
conda activate scgpt_mapt
```

## 1. Setup and Imports

All imports are consolidated here. Warnings are suppressed globally to prevent sklearn binning warnings from flooding the notebook output — these are cosmetic FutureWarnings that do not affect results.

In [ ]:
import os
import sys
import json
import time
import random
import warnings
warnings.filterwarnings('ignore')  # suppress sklearn FutureWarnings from binning

import numpy as np
import torch
import torch.nn as nn
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for HPC
import matplotlib.pyplot as plt
import seaborn as sns

import anndata as ad
from scipy.sparse import issparse
from sklearn.preprocessing import KBinsDiscretizer, label_binarize
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score,
    classification_report, confusion_matrix, roc_auc_score
)
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import umap

# Set random seeds for full reproducibility
# Same seeds = same train/val/test split + same model initialization every run
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Auto-detect GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Set project working directory
os.chdir('/scratch/arumuganainar.d/projects/scgpt')
os.makedirs('figures', exist_ok=True)

print(f'Device: {device}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')
print(f'PyTorch: {torch.__version__}')
print(f'Working dir: {os.getcwd()}')

## 2. Load Preprocessed Data and Model Configuration

We load two things here:

**Preprocessed neurons** (`neurons_hvg_preprocessed.h5ad`) — output of `scgpt_preprocessing.ipynb` containing 12,331 EX+INH neurons with 3,001 HVG genes and tangle stage labels.

**scGPT model configuration** — `args.json` defines the pretrained model architecture (embedding size, layers, attention heads) and `vocab.json` maps gene names to integer token IDs. Both are required to correctly tokenize our data and build a compatible model.

In [ ]:
def load_data_and_config(data_path, model_dir):
    """
    Load preprocessed AnnData and scGPT model configuration.

    Parameters
    ----------
    data_path : str
        Path to preprocessed h5ad file
    model_dir : str
        Directory containing args.json and vocab.json

    Returns
    -------
    adata : AnnData
        Preprocessed neuron subset
    model_args : dict
        Model architecture configuration
    vocab : dict
        Gene name to token ID mapping
    """
    # Load preprocessed neurons
    adata = ad.read_h5ad(data_path)

    # Load model architecture config
    with open(f'{model_dir}/args.json') as f:
        model_args = json.load(f)

    # Load gene vocabulary — maps gene names to integer token IDs
    with open(f'{model_dir}/vocab.json') as f:
        vocab = json.load(f)

    return adata, model_args, vocab


MODEL_DIR = './scgpt_human_model'
PAD_TOKEN = '<pad>'
PAD_VALUE = -2  # from args.json — value used for padding positions

adata, model_args, vocab = load_data_and_config(
    'neurons_hvg_preprocessed.h5ad', MODEL_DIR
)

# Verify data
print(f'Cells: {adata.n_obs:,} | Genes: {adata.n_vars:,}')
print(f'MAPT in dataset: {"MAPT" in adata.var_names}')
print(f'\nTangle label distribution:')
print(adata.obs['tangle_label'].value_counts().sort_index())

# Verify gene coverage in scGPT vocabulary
genes_in_vocab = sum(1 for g in adata.var_names if g in vocab)
print(f'\nVocab size: {len(vocab):,} genes')
print(f'Our genes in vocab: {genes_in_vocab} / {adata.n_vars} ({genes_in_vocab/adata.n_vars*100:.1f}%)')
print(f'MAPT in vocab: {"MAPT" in vocab}')

# Key model architecture parameters
print(f'\nModel architecture:')
print(f'  Embedding size: {model_args["embsize"]}')
print(f'  Transformer layers: {model_args["nlayers"]}')
print(f'  Attention heads: {model_args["nheads"]}')
print(f'  Expression bins: {model_args["n_bins"]}')

## 3. Tokenization

scGPT treats each cell like a sentence where genes are words. The model cannot accept a raw continuous expression matrix — it requires discrete tokens.

**Three tokenization steps:**

1. **Vocab filtering** — keep only genes that exist in the scGPT vocabulary (2,316/3,001 genes). Genes not in vocab have no pretrained embedding and are discarded.

2. **Expression binning** — convert continuous log-normalized expression values into 51 discrete bins, matching the pretraining format. Each gene is binned independently so bin values are relative to that gene's own expression distribution.

3. **Top-N selection** — for each cell, select the 200 most highly expressed genes as tokens. This limits sequence length for computational efficiency while retaining the most informative genes.

**Why 200 tokens:** Standard choice for scGPT fine-tuning. Most cells in our HVG subset express fewer than 200 genes, so little information is lost.

In [ ]:
def tokenize_expression(
    adata, vocab, n_bins=51, n_tokens=200
):
    """
    Convert expression matrix to scGPT token format.

    For each cell, selects the top n_tokens expressed genes,
    maps gene names to vocabulary IDs, and discretizes expression
    values into n_bins bins.

    Parameters
    ----------
    adata : AnnData
        Preprocessed neurons with tangle_label in obs
    vocab : dict
        Gene name to token ID mapping from scGPT
    n_bins : int
        Number of expression bins (must match pretraining, default 51)
    n_tokens : int
        Number of genes per cell to use as tokens (default 200)

    Returns
    -------
    all_gene_tokens : np.ndarray, shape (n_cells, n_tokens)
        Token IDs for selected genes per cell
    all_expr_tokens : np.ndarray, shape (n_cells, n_tokens)
        Binned expression values for selected genes per cell
    all_labels : np.ndarray, shape (n_cells,)
        Integer tangle stage labels
    """
    # Step 1 — filter to genes in scGPT vocabulary
    gene_names = adata.var_names.tolist()
    valid_mask = [g in vocab for g in gene_names]
    valid_genes = [g for g in gene_names if g in vocab]
    adata_valid = adata[:, valid_mask].copy()
    print(f'Genes in vocab: {len(valid_genes)} / {len(gene_names)}')

    # Convert to dense matrix
    expr_matrix = (
        adata_valid.X.toarray()
        if issparse(adata_valid.X)
        else adata_valid.X
    )
    print(f'Expression range: {expr_matrix.min():.3f} to {expr_matrix.max():.3f}')

    # Step 2 — bin each gene's expression into n_bins discrete levels
    # Each gene binned independently: bin 25 = medium for THAT gene
    print('Binning expression values...')
    binner = KBinsDiscretizer(
        n_bins=n_bins, encode='ordinal', strategy='quantile'
    )
    expr_binned = np.zeros_like(expr_matrix, dtype=np.int64)

    for i in range(expr_matrix.shape[1]):
        col = expr_matrix[:, i]
        nonzero_idx = col > 0
        # Only bin genes expressed in at least 10 cells
        if nonzero_idx.sum() > 10:
            col_nonzero = col[nonzero_idx].reshape(-1, 1)
            binned = (
                binner.fit_transform(col_nonzero)
                .flatten().astype(np.int64) + 1
            )
            expr_binned[nonzero_idx, i] = binned

    print(f'Binned range: {expr_binned.min()} to {expr_binned.max()}')

    # Step 3 — convert gene names to token IDs
    gene_token_ids = np.array([vocab[g] for g in valid_genes])

    # Step 4 — for each cell, select top n_tokens expressed genes
    print(f'Selecting top {n_tokens} genes per cell...')
    all_gene_tokens, all_expr_tokens, all_labels = [], [], []
    labels = adata.obs['tangle_label'].values.astype(np.int64)

    for i in range(adata.n_obs):
        cell_expr = expr_binned[i]
        nonzero_idx = np.where(cell_expr > 0)[0]

        gene_tok = np.zeros(n_tokens, dtype=np.int64)
        expr_tok = np.zeros(n_tokens, dtype=np.int64)

        if len(nonzero_idx) == 0:
            pass  # all zeros — padding only
        elif len(nonzero_idx) <= n_tokens:
            # Fewer genes than budget — use all, pad the rest
            gene_tok[:len(nonzero_idx)] = gene_token_ids[nonzero_idx]
            expr_tok[:len(nonzero_idx)] = cell_expr[nonzero_idx]
        else:
            # More genes than budget — keep highest expressed
            top_idx = np.argsort(cell_expr[nonzero_idx])[-n_tokens:]
            selected = nonzero_idx[top_idx]
            gene_tok = gene_token_ids[selected]
            expr_tok = cell_expr[selected]

        all_gene_tokens.append(gene_tok)
        all_expr_tokens.append(expr_tok)
        all_labels.append(labels[i])

    all_gene_tokens = np.array(all_gene_tokens)
    all_expr_tokens = np.array(all_expr_tokens)
    all_labels = np.array(all_labels)

    print(f'\nTokenized dataset shape: {all_gene_tokens.shape}')
    print(f'Label distribution: {np.bincount(all_labels)}')

    return all_gene_tokens, all_expr_tokens, all_labels


all_gene_tokens, all_expr_tokens, all_labels = tokenize_expression(
    adata, vocab,
    n_bins=model_args['n_bins'],  # 51 — must match pretraining
    n_tokens=200
)

## 4. Dataset and DataLoaders

**TangleStageDataset** wraps our numpy arrays in PyTorch's Dataset interface, returning a dictionary of tensors per cell matching the format expected by scGPT's forward pass.

**Stratified split** ensures each of the four tangle stages is proportionally represented in train, validation, and test sets — critical because Stage 6 makes up 52% of the data.

**WeightedRandomSampler** oversamples minority classes so each training batch has balanced class representation. Without this, Stage 6 would dominate every batch and the model would learn to always predict Stage 6.

**Split:** 80% train / 10% validation / 10% test

In [ ]:
class TangleStageDataset(Dataset):
    """
    PyTorch Dataset for scGPT tangle stage classification.

    Returns a dictionary with gene token IDs, binned expression values,
    and integer tangle stage labels for each cell.

    Parameters
    ----------
    gene_tokens : np.ndarray, shape (n_cells, n_tokens)
        Vocabulary token IDs for top expressed genes
    expr_tokens : np.ndarray, shape (n_cells, n_tokens)
        Binned expression values for top expressed genes
    labels : np.ndarray, shape (n_cells,)
        Integer tangle stage labels (0=Stage1, 1=Stage2, 2=Stage5, 3=Stage6)
    """
    def __init__(self, gene_tokens, expr_tokens, labels):
        self.gene_tokens = torch.LongTensor(gene_tokens)
        self.expr_tokens = torch.LongTensor(expr_tokens)
        self.labels = torch.LongTensor(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'gene_ids': self.gene_tokens[idx],
            'values':   self.expr_tokens[idx],
            'label':    self.labels[idx]
        }


def build_dataloaders(all_gene_tokens, all_expr_tokens, all_labels, seed=42):
    """
    Create stratified train/val/test splits and weighted DataLoaders.

    Parameters
    ----------
    all_gene_tokens, all_expr_tokens, all_labels : np.ndarray
        Full tokenized dataset
    seed : int
        Random seed for reproducible splits

    Returns
    -------
    train_loader, val_loader, test_loader : DataLoader
    train_idx, val_idx, test_idx : np.ndarray
        Cell indices for each split
    """
    idx = np.arange(len(all_labels))

    # Stratified split preserves class proportions in each subset
    train_idx, temp_idx = train_test_split(
        idx, test_size=0.2, random_state=seed, stratify=all_labels
    )
    val_idx, test_idx = train_test_split(
        temp_idx, test_size=0.5,
        random_state=seed, stratify=all_labels[temp_idx]
    )

    # Weighted sampler: oversample minority classes in training
    # Inverse frequency weighting — rare stages sampled more often
    train_labels = all_labels[train_idx]
    class_counts = np.bincount(train_labels)
    sample_weights = (1.0 / class_counts)[train_labels]
    sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(train_labels),
        replacement=True
    )

    # Build datasets
    train_ds = TangleStageDataset(
        all_gene_tokens[train_idx],
        all_expr_tokens[train_idx],
        all_labels[train_idx]
    )
    val_ds = TangleStageDataset(
        all_gene_tokens[val_idx],
        all_expr_tokens[val_idx],
        all_labels[val_idx]
    )
    test_ds = TangleStageDataset(
        all_gene_tokens[test_idx],
        all_expr_tokens[test_idx],
        all_labels[test_idx]
    )

    train_loader = DataLoader(train_ds, batch_size=32,
                               sampler=sampler, num_workers=2)
    val_loader   = DataLoader(val_ds,   batch_size=64,
                               shuffle=False, num_workers=2)
    test_loader  = DataLoader(test_ds,  batch_size=64,
                               shuffle=False, num_workers=2)

    return train_loader, val_loader, test_loader, train_idx, val_idx, test_idx


train_loader, val_loader, test_loader, train_idx, val_idx, test_idx = \
    build_dataloaders(all_gene_tokens, all_expr_tokens, all_labels, seed=SEED)

print(f'Train: {len(train_idx):,} cells ({len(train_loader)} batches)')
print(f'Val:   {len(val_idx):,} cells ({len(val_loader)} batches)')
print(f'Test:  {len(test_idx):,} cells ({len(test_loader)} batches)')

# Verify sampler is balancing classes
sample_batch = next(iter(train_loader))
print(f'\nSample batch label counts: {torch.bincount(sample_batch["label"])}')
print('(Should be roughly balanced across 4 classes)')

## 5. Load Pretrained scGPT and Configure Fine-tuning

**Model architecture** matches the pretrained checkpoint exactly — embedding size, layers, heads, and input format are all read from `args.json`.

**`strict=False` when loading weights** — allows mismatched classification head layers. The pretrained model has a different number of output classes. Our classification head (34 missing keys) gets randomly initialized and trained from scratch.

**Freezing strategy — critical design decision:**

We freeze most of the 51M pretrained parameters and only train 3.7M:
- Frozen: transformer layers 0-9 (general biological knowledge)
- Trainable: transformer layers 10-11 (task adaptation)
- Trainable: classification decoder (maps 512-dim embedding to 4 classes)

**Why unfreeze the last 2 layers:** Early transformer layers learn general gene co-expression patterns. Later layers learn task-specific representations. Unfreezing layers 10-11 allows them to re-weight attention toward disease-relevant gene relationships without destroying the foundational biological knowledge in layers 0-9.

**Why not freeze everything:** Initial experiment froze all layers except the classifier — the model predicted the same class for every cell (identical logits). The frozen transformer produced static embeddings that provided no useful signal for disease staging.

In [ ]:
from scgpt.model import TransformerModel


def build_and_load_model(model_args, vocab, model_dir, device, n_cls=4):
    """
    Build scGPT TransformerModel and load pretrained weights.

    Architecture parameters are read directly from args.json to ensure
    compatibility with the pretrained checkpoint.

    Parameters
    ----------
    model_args : dict
        Architecture config from args.json
    vocab : dict
        Gene vocabulary
    model_dir : str
        Path to pretrained model folder
    device : torch.device
        Target device
    n_cls : int
        Number of output classes (default 4 for tangle stages)

    Returns
    -------
    model : TransformerModel
        Model with pretrained weights loaded, moved to device
    """
    model = TransformerModel(
        ntoken=len(vocab),
        d_model=model_args['embsize'],          # 512
        nhead=model_args['nheads'],              # 8
        d_hid=model_args['d_hid'],               # 512
        nlayers=model_args['nlayers'],           # 12
        nlayers_cls=3,                           # classification head depth
        n_cls=n_cls,                             # 4 tangle stages
        vocab=vocab,
        dropout=model_args['dropout'],           # 0.2
        pad_token=PAD_TOKEN,
        pad_value=PAD_VALUE,
        do_mvc=False,
        do_dab=False,
        use_batch_labels=False,
        domain_spec_batchnorm=False,
        input_emb_style='continuous',
        n_input_bins=model_args['n_bins'],       # 51
        cell_emb_style='cls',                   # CLS token aggregation
        ecs_threshold=0.0,
        explicit_zero_prob=False,
        use_fast_transformer=False,
        pre_norm=False,
    )

    # Load pretrained weights
    # strict=False: allows mismatched classification head
    # Missing keys = our new cls head (randomly initialized, will be trained)
    # Unexpected keys = pretrained layers we dont use
    weights = torch.load(
        f'{model_dir}/best_model.pt', map_location=device
    )
    missing, unexpected = model.load_state_dict(weights, strict=False)
    print(f'Pretrained weights loaded')
    print(f'  Missing keys (new cls head): {len(missing)}')
    print(f'  Unexpected keys (ignored):   {len(unexpected)}')

    return model.to(device)


def configure_finetuning(model, unfreeze_last_n_layers=2):
    """
    Configure which layers are trainable for fine-tuning.

    Freezes all pretrained transformer layers except the last n,
    plus the classification decoder.

    Rationale: Early layers encode general gene co-expression patterns.
    Later layers encode task-specific representations and benefit from
    adaptation to the disease staging task.

    Parameters
    ----------
    model : TransformerModel
    unfreeze_last_n_layers : int
        Number of final transformer layers to unfreeze (default 2)

    Returns
    -------
    model : TransformerModel
        Model with frozen/unfrozen layers configured
    """
    n_layers = model_args['nlayers']  # 12

    # Freeze all layers first
    for param in model.parameters():
        param.requires_grad = False

    # Unfreeze: last n transformer layers + classification decoder
    unfreeze_names = ['cls_decoder'] + [
        f'transformer_encoder.layers.{n_layers - 1 - i}'
        for i in range(unfreeze_last_n_layers)
    ]

    for name, param in model.named_parameters():
        if any(x in name for x in unfreeze_names):
            param.requires_grad = True

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen    = sum(p.numel() for p in model.parameters() if not p.requires_grad)

    print(f'Trainable parameters: {trainable:,} ({trainable/(trainable+frozen)*100:.1f}%)')
    print(f'Frozen parameters:    {frozen:,}')
    print(f'Unfrozen layers: {unfreeze_names}')

    return model


model = build_and_load_model(model_args, vocab, MODEL_DIR, device)
model = configure_finetuning(model, unfreeze_last_n_layers=2)

## 6. Training

**Loss function — weighted CrossEntropyLoss:**  
Class weights are inversely proportional to class frequency. Stage 2 (rarest, n=1,532) gets weight 2.01 — errors on rare stages are penalized more heavily. Stage 6 weight was tripled (3x multiplier) after the first training run produced zero Stage 6 predictions.

**Optimizer — AdamW:**  
Standard choice for transformer fine-tuning. Weight decay=0.01 adds L2 regularization to prevent overfitting on our relatively small dataset (9,864 training cells).

**Learning rate — 1e-4:**  
Lower than typical ML training because we are fine-tuning a pretrained model. Too high a learning rate risks destroying the pretrained knowledge.

**Gradient clipping at 1.0:**  
Prevents exploding gradients — a common instability in transformer training. Ensures stable weight updates.

**Early stopping (patience=3):**  
Saves the best model checkpoint whenever validation loss improves. Stops training if no improvement for 3 consecutive epochs.

In [ ]:
def run_forward_pass(model, batch, vocab, pad_token, device):
    """
    Run a single forward pass through scGPT.

    The padding mask tells the model which positions contain real genes
    vs zero-padding. CLS=True activates the classification head.
    Other flags (CCE, MVC, ECS) are pretraining objectives — disabled.

    Parameters
    ----------
    model : TransformerModel
    batch : dict
        Dictionary with gene_ids, values, label tensors
    vocab : dict
    pad_token : str
    device : torch.device

    Returns
    -------
    logits : torch.Tensor, shape (batch_size, 4)
        Classification logits for 4 tangle stages
    labels : torch.Tensor, shape (batch_size,)
    """
    gene_ids = batch['gene_ids'].to(device)
    values   = batch['values'].to(device).float()
    labels   = batch['label'].to(device)

    output = model(
        gene_ids, values,
        src_key_padding_mask=gene_ids.eq(vocab.get(pad_token, 0)),
        batch_labels=None,
        CLS=True, CCE=False, MVC=False, ECS=False, do_sample=False,
    )

    # output['cls_output'] shape: (batch_size, n_cls=4)
    # This is already the final classification logits
    return output['cls_output'], labels


def train_model(
    model, train_loader, val_loader,
    criterion, optimizer, scheduler,
    vocab, pad_token, device,
    n_epochs=10, patience=3,
    checkpoint_path='scgpt_mapt_best.pt'
):
    """
    Full training loop with validation and early stopping.

    Saves best model checkpoint whenever validation loss improves.
    Stops training early if validation loss does not improve for
    `patience` consecutive epochs.

    Parameters
    ----------
    model : TransformerModel
    train_loader, val_loader : DataLoader
    criterion : loss function
    optimizer, scheduler : torch optimizers
    vocab : dict
    pad_token : str
    device : torch.device
    n_epochs : int
    patience : int
        Early stopping patience
    checkpoint_path : str
        Path to save best model weights

    Returns
    -------
    history : dict
        Training metrics per epoch
    """
    best_val_loss = float('inf')
    patience_counter = 0
    history = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_f1': []}

    for epoch in range(n_epochs):
        start = time.time()

        # --- Training phase ---
        model.train()
        train_loss, correct, total = 0.0, 0, 0

        for batch in train_loader:
            optimizer.zero_grad()
            logits, labels = run_forward_pass(
                model, batch, vocab, pad_token, device
            )
            loss = criterion(logits, labels)
            loss.backward()
            # Clip gradients to prevent exploding updates
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            train_loss += loss.item()
            correct += (logits.argmax(1) == labels).sum().item()
            total   += labels.size(0)

        avg_train_loss = train_loss / len(train_loader)

        # --- Validation phase ---
        model.eval()
        val_loss, all_preds, all_true = 0.0, [], []

        with torch.no_grad():
            for batch in val_loader:
                logits, labels = run_forward_pass(
                    model, batch, vocab, pad_token, device
                )
                val_loss += criterion(logits, labels).item()
                all_preds.extend(logits.argmax(1).cpu().numpy())
                all_true.extend(labels.cpu().numpy())

        avg_val_loss = val_loss / len(val_loader)
        val_acc = accuracy_score(all_true, all_preds)
        val_f1  = f1_score(all_true, all_preds, average='weighted')
        pred_dist = np.bincount(all_preds, minlength=4).tolist()

        scheduler.step(avg_val_loss)
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['val_acc'].append(val_acc)
        history['val_f1'].append(val_f1)

        elapsed = time.time() - start
        print(
            f'Epoch {epoch+1}/{n_epochs} | '
            f'Train Loss: {avg_train_loss:.4f} | '
            f'Val Loss: {avg_val_loss:.4f} | '
            f'Val Acc: {val_acc:.4f} | '
            f'Val F1: {val_f1:.4f} | '
            f'Pred dist: {pred_dist} | '
            f'Time: {elapsed:.1f}s'
        )

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), checkpoint_path)
            print(f'  -> Best model saved (val_loss={best_val_loss:.4f})')
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'Early stopping at epoch {epoch+1}')
                break

    return history


# Weighted CrossEntropyLoss
# Inverse frequency weighting + 3x boost for Stage 6
# Stage 6 boost was necessary after first run produced zero Stage 6 predictions
counts = np.bincount(all_labels)
class_weights = torch.FloatTensor([
    len(all_labels) / (4 * counts[0]),        # Stage 1: 1.61
    len(all_labels) / (4 * counts[1]),        # Stage 2: 2.01
    len(all_labels) / (4 * counts[2]),        # Stage 5: 1.26
    len(all_labels) / (4 * counts[3]) * 3.0  # Stage 6: 1.44 * 3
]).to(device)

criterion  = nn.CrossEntropyLoss(weight=class_weights)
optimizer  = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4, weight_decay=0.01
)
scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2
)

print('Class weights:', class_weights.cpu().numpy().round(3))
print('\nStarting training...')
print('=' * 70)

history = train_model(
    model, train_loader, val_loader,
    criterion, optimizer, scheduler,
    vocab, PAD_TOKEN, device,
    n_epochs=15, patience=3,
    checkpoint_path='scgpt_mapt_best.pt'
)

## 7. Evaluation

We evaluate the best saved checkpoint on the held-out test set — cells the model has never seen during training or validation.

**Metrics:**
- **Accuracy** — overall fraction of correct predictions
- **Per-class F1** — harmonic mean of precision and recall for each stage
- **Weighted AUC** — area under ROC curve, weighted by class frequency. Above 0.5 = better than random; 0.68 indicates meaningful discriminative power.
- **Confusion matrix** — which stages are confused with which

**Results summary:**
- Overall accuracy: 48% (vs 25% random baseline on 4 classes)
- Weighted AUC: 0.68
- Stage 6 F1: 0.63 (most discriminable — end-stage pathology is transcriptionally distinct)
- Stage 5 F1: 0.21 (hardest — consistent with transitional biology between severe and end-stage)

In [ ]:
def evaluate_model(model, loader, vocab, pad_token, device, return_probs=False):
    """
    Evaluate model on a DataLoader and return predictions and optionally probabilities.

    Parameters
    ----------
    model : TransformerModel
    loader : DataLoader
    vocab : dict
    pad_token : str
    device : torch.device
    return_probs : bool
        If True, also return softmax probabilities for AUC calculation

    Returns
    -------
    all_preds : np.ndarray
    all_true : np.ndarray
    all_probs : np.ndarray or None
    """
    model.eval()
    all_preds, all_true, all_probs = [], [], []

    with torch.no_grad():
        for batch in loader:
            logits, labels = run_forward_pass(
                model, batch, vocab, pad_token, device
            )
            all_preds.extend(logits.argmax(1).cpu().numpy())
            all_true.extend(labels.cpu().numpy())
            if return_probs:
                probs = torch.softmax(logits, dim=1)
                all_probs.extend(probs.cpu().numpy())

    all_preds = np.array(all_preds)
    all_true  = np.array(all_true)
    all_probs = np.array(all_probs) if return_probs else None

    return all_preds, all_true, all_probs


def plot_confusion_matrix(all_true, all_preds, save_path):
    """
    Plot and save confusion matrix heatmap.

    Parameters
    ----------
    all_true, all_preds : np.ndarray
    save_path : str
    """
    stage_labels = ['Stage 1', 'Stage 2', 'Stage 5', 'Stage 6']
    cm = confusion_matrix(all_true, all_preds)

    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(
        cm, annot=True, fmt='d', ax=ax,
        xticklabels=stage_labels,
        yticklabels=stage_labels,
        cmap='Blues'
    )
    ax.set_xlabel('Predicted Stage')
    ax.set_ylabel('True Stage')
    ax.set_title('Confusion Matrix — scGPT Tangle Stage Classification')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    print(f'Saved: {save_path}')


# Load best checkpoint
model.load_state_dict(
    torch.load('scgpt_mapt_best.pt', map_location=device)
)
print('Best model loaded')

# Evaluate on test set
test_preds, test_true, test_probs = evaluate_model(
    model, test_loader, vocab, PAD_TOKEN, device, return_probs=True
)

# Classification report
print('\n=== FINAL TEST SET RESULTS ===')
print(classification_report(
    test_true, test_preds,
    target_names=['Stage 1', 'Stage 2', 'Stage 5', 'Stage 6']
))

# Weighted AUC
test_true_bin = label_binarize(test_true, classes=[0, 1, 2, 3])
auc = roc_auc_score(
    test_true_bin, test_probs,
    multi_class='ovr', average='weighted'
)
print(f'Weighted AUC: {auc:.4f}')

# Confusion matrix
plot_confusion_matrix(
    test_true, test_preds,
    save_path='figures/confusion_matrix_test.png'
)

**Test set interpretation:**

- **Stage 6 (F1=0.63):** End-stage tau pathology produces the most distinct transcriptional signature. The model reliably identifies cells from patients with the highest tangle burden.
- **Stage 5 (F1=0.21):** The weakest performer. Stage 5 may represent a transcriptional transition state where cells share features with both Stage 2 and Stage 6 — this is biologically interpretable rather than a model failure.
- **Stage 1/Stage 6 confusion:** 79 Stage 1 cells are predicted as Stage 6. This is consistent with the survivor bias hypothesis — the most resilient Stage 6 neurons may share transcriptional profiles with Stage 1 neurons that have not yet mounted a stress response.
- **Overall 48% accuracy on 4 classes** represents nearly 2x random baseline (25%), with AUC=0.68 indicating real discriminative power beyond chance.

## 8. Embedding Visualization and Results

We extract the 512-dimensional cell embeddings produced by the fine-tuned scGPT and project them to 2D using UMAP. This produces two key figures:

1. **PCA vs scGPT UMAP comparison** — demonstrates that the foundation model learns a qualitatively different representation of cellular state compared to standard PCA
2. **Prediction correctness map** — spatial clustering of correct predictions (blue) confirms the embeddings contain real biological signal even where UMAP compression loses resolution

In [ ]:
def extract_embeddings(model, all_gene_tokens, all_expr_tokens,
                        all_labels, vocab, pad_token, device, batch_size=64):
    """
    Extract 512-dimensional cell embeddings from fine-tuned scGPT.

    Uses output['cell_emb'] — the CLS token representation after
    all 12 transformer layers. This 512-dim vector is the model's
    learned summary of each cell's transcriptional state.

    Parameters
    ----------
    model : TransformerModel
    all_gene_tokens, all_expr_tokens, all_labels : np.ndarray
        Full tokenized dataset
    vocab : dict
    pad_token : str
    device : torch.device
    batch_size : int

    Returns
    -------
    embeddings : np.ndarray, shape (n_cells, 512)
    labels : np.ndarray, shape (n_cells,)
    """
    full_ds = TangleStageDataset(
        all_gene_tokens, all_expr_tokens, all_labels
    )
    loader = DataLoader(full_ds, batch_size=batch_size, shuffle=False)

    model.eval()
    all_embs, all_labs = [], []

    with torch.no_grad():
        for batch in loader:
            gene_ids = batch['gene_ids'].to(device)
            values   = batch['values'].to(device).float()

            output = model(
                gene_ids, values,
                src_key_padding_mask=gene_ids.eq(vocab.get(pad_token, 0)),
                batch_labels=None,
                CLS=True, CCE=False, MVC=False, ECS=False, do_sample=False,
            )
            # cell_emb: 512-dim CLS token embedding
            all_embs.extend(output['cell_emb'].cpu().numpy())
            all_labs.extend(batch['label'].numpy())

    return np.array(all_embs), np.array(all_labs)


def plot_embedding_analysis(embedding_2d, all_labs, all_preds_full,
                             adata, save_dir='figures'):
    """
    Generate two-panel embedding figures:
    1. PCA UMAP vs scGPT UMAP colored by tangle stage
    2. scGPT UMAP colored by prediction correctness

    Parameters
    ----------
    embedding_2d : np.ndarray, shape (n_cells, 2)
        UMAP projection of scGPT embeddings
    all_labs : np.ndarray
        True tangle stage labels
    all_preds_full : np.ndarray
        Model predictions for all cells
    adata : AnnData
        Preprocessed data with PCA UMAP in obsm['X_umap']
    save_dir : str
    """
    stage_names = {0: 'Stage 1', 1: 'Stage 2', 2: 'Stage 5', 3: 'Stage 6'}
    colors      = ['#2196F3', '#4CAF50', '#FF9800', '#F44336']
    correct     = all_preds_full == all_labs

    # Figure 1 — PCA vs scGPT UMAP
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    pca_umap  = adata.obsm['X_umap']

    for ax, coords, title in zip(
        axes,
        [embedding_2d, pca_umap],
        ['scGPT Embeddings UMAP\n(after fine-tuning)',
         'PCA UMAP\n(before fine-tuning)']
    ):
        for stage, color in zip([0, 1, 2, 3], colors):
            mask = all_labs == stage
            ax.scatter(
                coords[mask, 0], coords[mask, 1],
                c=color, label=stage_names[stage], alpha=0.4, s=5
            )
        ax.set_title(title, fontsize=13)
        ax.legend(markerscale=3)
        ax.set_xlabel('UMAP1')
        ax.set_ylabel('UMAP2')

    plt.suptitle(
        'MAPT Tangle Stage: PCA vs scGPT Embeddings', fontsize=14
    )
    plt.tight_layout()
    path1 = f'{save_dir}/UMAP_comparison_scGPT_vs_PCA.png'
    plt.savefig(path1, dpi=150, bbox_inches='tight')
    print(f'Saved: {path1}')

    # Figure 2 — Prediction correctness
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Left: true stage labels
    for stage, color in zip([0, 1, 2, 3], colors):
        mask = all_labs == stage
        axes[0].scatter(
            embedding_2d[mask, 0], embedding_2d[mask, 1],
            c=color, label=stage_names[stage], alpha=0.4, s=5
        )
    axes[0].set_title('scGPT UMAP — True Tangle Stage', fontsize=13)
    axes[0].legend(markerscale=3)

    # Right: correct vs incorrect
    axes[1].scatter(
        embedding_2d[correct, 0], embedding_2d[correct, 1],
        c='#2196F3', label='Correct', alpha=0.4, s=5
    )
    axes[1].scatter(
        embedding_2d[~correct, 0], embedding_2d[~correct, 1],
        c='#F44336', label='Incorrect', alpha=0.4, s=5
    )
    axes[1].set_title(
        'scGPT UMAP — Prediction Correctness\n(Blue=Correct, Red=Incorrect)',
        fontsize=13
    )
    axes[1].legend(markerscale=3)

    plt.suptitle('scGPT Embedding Analysis', fontsize=14)
    plt.tight_layout()
    path2 = f'{save_dir}/UMAP_correctness.png'
    plt.savefig(path2, dpi=150, bbox_inches='tight')
    print(f'Saved: {path2}')
    print(f'Overall accuracy: {correct.mean():.3f}')


# Extract embeddings
print('Extracting scGPT cell embeddings...')
embeddings, emb_labels = extract_embeddings(
    model, all_gene_tokens, all_expr_tokens, all_labels,
    vocab, PAD_TOKEN, device
)
print(f'Embeddings shape: {embeddings.shape}')

# Project to 2D with UMAP
print('Running UMAP on 512-dim embeddings...')
reducer = umap.UMAP(n_neighbors=15, min_dist=0.3, random_state=SEED)
embedding_2d = reducer.fit_transform(embeddings)
print('UMAP complete')

# Get predictions for all cells
all_preds_full, _, _ = evaluate_model(
    model,
    DataLoader(
        TangleStageDataset(all_gene_tokens, all_expr_tokens, all_labels),
        batch_size=64, shuffle=False
    ),
    vocab, PAD_TOKEN, device
)

# Generate figures
plot_embedding_analysis(
    embedding_2d, emb_labels, all_preds_full, adata
)

**Embedding interpretation:**

The scGPT UMAP appears more diffuse than the PCA UMAP — this is expected and not a failure. PCA separates cells primarily by **cell type** (EX vs INH neurons form distinct clusters). scGPT embeddings represent cells by **disease-relevant transcriptional state**, which is a more continuous signal — tau pathology progression does not have sharp transcriptional boundaries between stages.

The prediction correctness map (right panel) confirms real biological signal: correct predictions spatially cluster in specific embedding regions rather than being randomly distributed. This demonstrates the embeddings encode meaningful disease information that survives UMAP compression.

## 9. Save Results Summary

In [ ]:
def save_results(
    test_true, test_preds, test_probs, history, save_path='results_summary.json'
):
    """
    Compute and save quantitative results to JSON.

    All metrics are computed dynamically from actual predictions
    rather than hardcoded, ensuring the file stays accurate
    if training is re-run.

    Parameters
    ----------
    test_true, test_preds, test_probs : np.ndarray
    history : dict
        Training metrics from train_model()
    save_path : str
    """
    stage_names = ['Stage1', 'Stage2', 'Stage5', 'Stage6']

    # Per-class F1
    f1_per_class = f1_score(test_true, test_preds, average=None)

    # Weighted AUC
    test_true_bin = label_binarize(test_true, classes=[0, 1, 2, 3])
    auc = roc_auc_score(
        test_true_bin, test_probs,
        multi_class='ovr', average='weighted'
    )

    results = {
        'model': 'scGPT_human fine-tuned',
        'task': 'Tangle Stage Classification (4-class)',
        'dataset': 'Morabito et al. 2021 GSE174367',
        'n_cells_total': 12331,
        'n_cells_test': len(test_true),
        'cell_types': 'EX + INH neurons only',
        'test_accuracy': round(accuracy_score(test_true, test_preds), 4),
        'weighted_auc': round(float(auc), 4),
        'weighted_f1': round(
            float(f1_score(test_true, test_preds, average='weighted')), 4
        ),
        'per_class_f1': {
            s: round(float(f), 4)
            for s, f in zip(stage_names, f1_per_class)
        },
        'training_config': {
            'pretrained_model': 'scGPT_human (33M normal human cells)',
            'unfrozen_layers': 'transformer_encoder.layers.10, .11 + cls_decoder',
            'trainable_params': 3685380,
            'frozen_params': 47648257,
            'optimizer': 'AdamW',
            'learning_rate': '1e-4',
            'class_weights': 'inverse frequency + 3x Stage6 boost',
            'n_tokens_per_cell': 200,
            'n_bins': 51,
            'epochs_trained': len(history['val_loss'])
        },
        'key_biological_findings': [
            'Stage 6 most discriminable (F1=0.63) — end-stage tau pathology is transcriptionally distinct',
            'Stage 5 hardest (F1=0.21) — consistent with transitional biology between severe and end-stage',
            'Stage 1/6 confusion consistent with survivor bias: resilient Stage 6 neurons resemble Stage 1',
            'MAPT peaks at Stage 2 then declines — vulnerable high-MAPT neurons die before late-stage sampling'
        ]
    }

    with open(save_path, 'w') as f:
        json.dump(results, f, indent=2)

    print(f'Results saved to {save_path}')
    print(f'\nKey metrics:')
    print(f'  Test accuracy: {results["test_accuracy"]}')
    print(f'  Weighted AUC:  {results["weighted_auc"]}')
    print(f'  Weighted F1:   {results["weighted_f1"]}')
    print(f'  Per-class F1:  {results["per_class_f1"]}')


save_results(test_true, test_preds, test_probs, history)

print('\n=== Project complete ===')
print('Output files:')
for f in sorted(os.listdir('figures')):
    print(f'  figures/{f}')
print('  scgpt_mapt_best.pt')
print('  results_summary.json')